# Chapter 08 – Data Modification
## Individual Assignment – 10 Propositions & Queries
**Database:** Northwinds2024 (TSQLV4 / Sales Schema)  
**Reference:** T-SQL Fundamentals – Itzik Ben-Gan, Chapter 08  
**File name format:** `Individual_Group1_HW8_MemberName.ipynb`

---
### Setup
Run the cell below once to establish the database connection.  
Replace the connection string values with your own server/database details.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install pyodbc pandas tabulate openpyxl

import pyodbc
import pandas as pd
from tabulate import tabulate

# ── Connection ────────────────────────────────────────────────────────────────
SERVER   = r'YOUR_SERVER_NAME'   # e.g. localhost\SQLEXPRESS
DATABASE = 'Northwinds2024'      # your restored .bak database name

conn_str = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER};'
    f'DATABASE={DATABASE};'
    f'Trusted_Connection=yes;'
)

conn = pyodbc.connect(conn_str)
print(f'Connected to [{DATABASE}] on [{SERVER}]  ✓')

# Helper – run a SELECT and display as a formatted table
def run_query(sql: str, title: str = '') -> pd.DataFrame:
    df = pd.read_sql(sql, conn)
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(tabulate(df, headers='keys', tablefmt='psql', showindex=False))
    print(f'({len(df)} row(s) returned)\n')
    return df

# Helper – run a DML statement and report rows affected
def run_dml(sql: str, title: str = '') -> None:
    cursor = conn.cursor()
    cursor.execute(sql)
    conn.commit()
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(f'  {cursor.rowcount} row(s) affected.\n')

---
## Proposition 1
**Statement:** Create a working copy of the `Sales.Orders` table inside the `dbo` schema using a `SELECT INTO` statement, then verify the copy by counting its rows.

**Concept tested:** `SELECT INTO` – creates a new table and populates it in one step; useful for staging or backup copies.

In [ ]:
# ── Proposition 1 – SELECT INTO (create dbo.Orders from Sales.Orders) ─────────

drop_sql = "DROP TABLE IF EXISTS dbo.Orders;"

select_into_sql = """
SELECT orderid, orderdate, custid, empid,
       shipperid, freight, shipcountry, shipcity, shipregion
INTO   dbo.Orders
FROM   Sales.Orders;
"""

verify_sql = "SELECT COUNT(*) AS TotalOrders FROM dbo.Orders;"

run_dml(drop_sql,        'Step 1 – Drop existing dbo.Orders (if any)')
run_dml(select_into_sql, 'Step 2 – SELECT INTO dbo.Orders')
run_query(verify_sql,    'Step 3 – Verify row count')

---
## Proposition 2
**Statement:** Insert a new customer (custid = 100, 'Coho Winery', USA, WA, Redmond) into a `dbo.Customers` staging table using `INSERT VALUES`, then confirm the record exists.

**Concept tested:** `INSERT VALUES` with explicit column list.

In [ ]:
# ── Proposition 2 – INSERT VALUES ─────────────────────────────────────────────

setup_sql = """
DROP TABLE IF EXISTS dbo.Customers;
CREATE TABLE dbo.Customers
(
  custid      INT           NOT NULL PRIMARY KEY,
  companyname NVARCHAR(40)  NOT NULL,
  country     NVARCHAR(15)  NOT NULL,
  region      NVARCHAR(15)  NULL,
  city        NVARCHAR(15)  NOT NULL
);
"""

insert_sql = """
INSERT INTO dbo.Customers (custid, companyname, country, region, city)
VALUES (100, N'Coho Winery', N'USA', N'WA', N'Redmond');
"""

verify_sql = "SELECT * FROM dbo.Customers WHERE custid = 100;"

run_dml(setup_sql,  'Step 1 – Create dbo.Customers')
run_dml(insert_sql, 'Step 2 – INSERT single row')
run_query(verify_sql, 'Step 3 – Verify inserted row')

---
## Proposition 3
**Statement:** Populate `dbo.Customers` with all customers from `Sales.Customers` who have placed at least one order, using `INSERT SELECT`.

**Concept tested:** `INSERT SELECT` combined with a semi-join (`EXISTS`) to filter only customers that appear in `Sales.Orders`.

In [ ]:
# ── Proposition 3 – INSERT SELECT (customers who placed orders) ────────────────

insert_select_sql = """
INSERT INTO dbo.Customers (custid, companyname, country, region, city)
SELECT custid, companyname, country, region, city
FROM   Sales.Customers AS C
WHERE  EXISTS (
    SELECT 1
    FROM   Sales.Orders AS O
    WHERE  O.custid = C.custid
)
AND custid <> 100;   -- avoid PK conflict with row inserted in Prop 2
"""

verify_sql = """
SELECT country, COUNT(*) AS CustomerCount
FROM   dbo.Customers
GROUP  BY country
ORDER  BY CustomerCount DESC;
"""

run_dml(insert_select_sql, 'INSERT customers who placed orders')
run_query(verify_sql,      'Customers by country (verification)')

---
## Proposition 4
**Statement:** Delete all orders from `dbo.Orders` that were placed before August 2014, and use the `OUTPUT` clause to return the `orderid` and `orderdate` of each deleted row.

**Concept tested:** `DELETE … OUTPUT deleted.*` – captures the rows being removed in a single atomic statement.

In [ ]:
# ── Proposition 4 – DELETE with OUTPUT ────────────────────────────────────────

# We capture OUTPUT rows by inserting into a temp table, then SELECT from it
delete_output_sql = """
DROP TABLE IF EXISTS #DeletedOrders;
CREATE TABLE #DeletedOrders (orderid INT, orderdate DATE);

DELETE FROM dbo.Orders
OUTPUT deleted.orderid, deleted.orderdate
INTO   #DeletedOrders
WHERE  orderdate < '20140801';

SELECT orderid, orderdate
FROM   #DeletedOrders
ORDER  BY orderdate;
"""

df_deleted = run_query(delete_output_sql, 'Orders deleted (placed before Aug 2014)')

---
## Proposition 5
**Statement:** Delete from `dbo.Orders` all orders placed by customers whose country is Brazil, using a JOIN-based DELETE.

**Concept tested:** `DELETE` with a `FROM … JOIN` to filter rows based on a related table.

In [ ]:
# ── Proposition 5 – DELETE via JOIN (Brazil customers) ────────────────────────

# First, see how many Brazil orders exist before deletion
count_before_sql = """
SELECT COUNT(*) AS BrazilOrdersBefore
FROM   dbo.Orders AS O
JOIN   Sales.Customers AS C ON O.custid = C.custid
WHERE  C.country = N'Brazil';
"""

delete_brazil_sql = """
DELETE O
FROM   dbo.Orders AS O
JOIN   Sales.Customers AS C ON O.custid = C.custid
WHERE  C.country = N'Brazil';
"""

count_after_sql = """
SELECT COUNT(*) AS BrazilOrdersAfter
FROM   dbo.Orders AS O
JOIN   Sales.Customers AS C ON O.custid = C.custid
WHERE  C.country = N'Brazil';
"""

run_query(count_before_sql, 'Brazil orders BEFORE delete')
run_dml(delete_brazil_sql,  'DELETE Brazil orders')
run_query(count_after_sql,  'Brazil orders AFTER delete')

---
## Proposition 6
**Statement:** Update `dbo.Customers`: replace every NULL `region` value with the string `'<None>'`, and use the `OUTPUT` clause to show the `custid`, old region, and new region for each affected row.

**Concept tested:** `UPDATE … OUTPUT deleted.<col> AS old…, inserted.<col> AS new…`

In [ ]:
# ── Proposition 6 – UPDATE NULL region with OUTPUT ────────────────────────────

update_output_sql = """
DROP TABLE IF EXISTS #RegionChanges;
CREATE TABLE #RegionChanges (custid INT, oldregion NVARCHAR(15), newregion NVARCHAR(15));

UPDATE dbo.Customers
SET    region = N'<None>'
OUTPUT inserted.custid,
       deleted.region  AS oldregion,
       inserted.region AS newregion
INTO   #RegionChanges
WHERE  region IS NULL;

SELECT custid, oldregion, newregion
FROM   #RegionChanges
ORDER  BY custid;
"""

df_region = run_query(update_output_sql, 'Region NULL → <None> (with OUTPUT)')

---
## Proposition 7
**Statement:** Update shipping address fields (`shipcountry`, `shipregion`, `shipcity`) in `dbo.Orders` for all UK orders, setting them to the matching customer's current `country`, `region`, and `city` from `dbo.Customers`.

**Concept tested:** `UPDATE … FROM … JOIN` – updating one table's columns using values from another.

In [ ]:
# ── Proposition 7 – UPDATE ship address from Customers (UK orders) ─────────────

update_uk_sql = """
UPDATE O
SET    O.shipcountry = C.country,
       O.shipregion  = C.region,
       O.shipcity    = C.city
FROM   dbo.Orders    AS O
JOIN   dbo.Customers AS C ON O.custid = C.custid
WHERE  O.shipcountry = N'UK';
"""

verify_uk_sql = """
SELECT TOP 10
       O.orderid, O.custid, O.shipcountry, O.shipregion, O.shipcity
FROM   dbo.Orders AS O
JOIN   dbo.Customers AS C ON O.custid = C.custid
WHERE  C.country = N'UK'
ORDER  BY O.orderid;
"""

run_dml(update_uk_sql,  'UPDATE UK order shipping addresses')
run_query(verify_uk_sql,'Top 10 UK orders after update')

---
## Proposition 8
**Statement:** Use a `MERGE` statement to synchronise a `dbo.CustomersStage` (new/updated records) into `dbo.Customers`: update existing customers and insert new ones. Output the action taken (`INSERT` or `UPDATE`) along with old and new company names.

**Concept tested:** `MERGE … WHEN MATCHED … WHEN NOT MATCHED … OUTPUT $action`

In [ ]:
# ── Proposition 8 – MERGE with OUTPUT ─────────────────────────────────────────

stage_setup_sql = """
DROP TABLE IF EXISTS dbo.CustomersStage;
CREATE TABLE dbo.CustomersStage
(
  custid      INT           NOT NULL PRIMARY KEY,
  companyname NVARCHAR(40)  NOT NULL,
  country     NVARCHAR(15)  NOT NULL,
  region      NVARCHAR(15)  NULL,
  city        NVARCHAR(15)  NOT NULL
);
-- Existing customer to UPDATE
INSERT INTO dbo.CustomersStage VALUES (1, N'Updated Company NRZBB', N'Germany', NULL, N'Berlin');
-- New customer to INSERT
INSERT INTO dbo.CustomersStage VALUES (200, N'New Startup Inc.', N'USA', N'CA', N'San Francisco');
"""

merge_output_sql = """
DROP TABLE IF EXISTS #MergeLog;
CREATE TABLE #MergeLog
(
  theaction   NVARCHAR(10),
  custid      INT,
  oldname     NVARCHAR(40),
  newname     NVARCHAR(40)
);

MERGE INTO dbo.Customers AS TGT
USING dbo.CustomersStage AS SRC
    ON TGT.custid = SRC.custid
WHEN MATCHED THEN
    UPDATE SET TGT.companyname = SRC.companyname,
               TGT.country     = SRC.country,
               TGT.region      = SRC.region,
               TGT.city        = SRC.city
WHEN NOT MATCHED THEN
    INSERT (custid, companyname, country, region, city)
    VALUES (SRC.custid, SRC.companyname, SRC.country, SRC.region, SRC.city)
OUTPUT $action               AS theaction,
       inserted.custid,
       deleted.companyname   AS oldname,
       inserted.companyname  AS newname
INTO   #MergeLog;

SELECT * FROM #MergeLog;
"""

run_dml(stage_setup_sql, 'Setup staging table')
run_query(merge_output_sql, 'MERGE result log')

---
## Proposition 9
**Statement:** Perform a bulk insert of the `orders.txt` flat file into a temporary `dbo.BulkOrders` table using `BULK INSERT`, then display all loaded rows.

**Concept tested:** `BULK INSERT` with `FIELDTERMINATOR` and `ROWTERMINATOR` — efficient bulk data loading from a flat file.

> **Note:** Update the `@file_path` variable below to the actual path of `orders.txt` on your SQL Server.

In [ ]:
# ── Proposition 9 – BULK INSERT from orders.txt ────────────────────────────────
# Update file_path to the absolute path accessible by your SQL Server service account

file_path = r'C:\temp\orders.txt'   # <-- change this

bulk_setup_sql = """
DROP TABLE IF EXISTS dbo.BulkOrders;
CREATE TABLE dbo.BulkOrders
(
  orderid   INT  NOT NULL,
  orderdate DATE NOT NULL,
  empid     INT  NOT NULL,
  custid    INT  NOT NULL
);
"""

bulk_insert_sql = f"""
BULK INSERT dbo.BulkOrders
FROM '{file_path}'
WITH
(
    DATAFILETYPE   = 'char',
    FIELDTERMINATOR = ',',
    ROWTERMINATOR   = '\\n'
);
"""

verify_bulk_sql = "SELECT * FROM dbo.BulkOrders ORDER BY orderid;"

run_dml(bulk_setup_sql,  'Create dbo.BulkOrders')
run_dml(bulk_insert_sql, 'BULK INSERT from orders.txt')
run_query(verify_bulk_sql, 'All bulk-loaded rows')

---
## Proposition 10
**Statement:** Use `TRUNCATE TABLE` to efficiently empty both `dbo.OrderDetails` and `dbo.Orders` (respecting the FK dependency order), then confirm both tables are empty.

**Concept tested:** `TRUNCATE TABLE` vs `DELETE` — `TRUNCATE` is minimally logged and resets identity seeds, but requires dropping FK constraints first (or truncating in child → parent order).

In [ ]:
# ── Proposition 10 – TRUNCATE TABLE (child before parent) ─────────────────────

# Recreate dbo.Orders + dbo.OrderDetails with FK for a clean demo
recreate_sql = """
DROP TABLE IF EXISTS dbo.OrderDetails2, dbo.Orders2;

CREATE TABLE dbo.Orders2
(
    orderid   INT  NOT NULL CONSTRAINT PK_Orders2 PRIMARY KEY,
    orderdate DATE NOT NULL,
    custid    INT  NOT NULL
);

CREATE TABLE dbo.OrderDetails2
(
    orderid   INT   NOT NULL,
    productid INT   NOT NULL,
    qty       SMALLINT NOT NULL DEFAULT 1,
    CONSTRAINT PK_OrderDetails2 PRIMARY KEY (orderid, productid),
    CONSTRAINT FK_OD2_O2 FOREIGN KEY (orderid) REFERENCES dbo.Orders2(orderid)
);

INSERT INTO dbo.Orders2      VALUES (1, '2024-01-01', 10), (2, '2024-01-02', 20);
INSERT INTO dbo.OrderDetails2 VALUES (1, 101, 5), (2, 102, 3);
"""

count_before_sql = """
SELECT 'Orders2'      AS tbl, COUNT(*) AS rows FROM dbo.Orders2
UNION ALL
SELECT 'OrderDetails2',       COUNT(*)         FROM dbo.OrderDetails2;
"""

# Truncate child first, then parent
truncate_sql = """
TRUNCATE TABLE dbo.OrderDetails2;
TRUNCATE TABLE dbo.Orders2;
"""

count_after_sql = """
SELECT 'Orders2'      AS tbl, COUNT(*) AS rows FROM dbo.Orders2
UNION ALL
SELECT 'OrderDetails2',       COUNT(*)         FROM dbo.OrderDetails2;
"""

run_dml(recreate_sql,      'Setup demo tables with sample data')
run_query(count_before_sql,'Row counts BEFORE TRUNCATE')
run_dml(truncate_sql,      'TRUNCATE OrderDetails2, then Orders2')
run_query(count_after_sql, 'Row counts AFTER TRUNCATE')

---
## Cleanup
Run this cell after grading to remove all working tables created during the assignment.

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
cleanup_sql = """
DROP TABLE IF EXISTS dbo.OrderDetails2, dbo.Orders2;
DROP TABLE IF EXISTS dbo.BulkOrders;
DROP TABLE IF EXISTS dbo.CustomersStage;
DROP TABLE IF EXISTS dbo.Customers;
DROP TABLE IF EXISTS dbo.Orders;
"""
run_dml(cleanup_sql, 'All working tables dropped  ✓')
conn.close()
print('Connection closed.')

---
# Part 2 – Excel Power Query Report (5 Queries)

The cell below exports five query result sets as separate sheets into a single Excel workbook.  
Open the file in Excel and use **Data → Get Data → From Table/Range** on each sheet to load them into Power Query, then build your report.

| Sheet | Description |
|-------|-------------|
| Q1_OrdersByCountry | Total orders and freight grouped by ship country |
| Q2_TopCustomers | Top 20 customers by total order value |
| Q3_MonthlyRevenue | Monthly revenue trend (2014–2016) |
| Q4_EmployeePerformance | Orders and avg freight per employee |
| Q5_ProductSales | Units sold and revenue per product |


In [ ]:
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, PatternFill, Alignment

# Re-open connection if closed
conn = pyodbc.connect(conn_str)

# ── Query 1: Orders by Ship Country ──────────────────────────────────────────
q1 = pd.read_sql("""
SELECT shipcountry,
       COUNT(*)         AS TotalOrders,
       SUM(freight)     AS TotalFreight,
       AVG(freight)     AS AvgFreight
FROM   Sales.Orders
GROUP  BY shipcountry
ORDER  BY TotalOrders DESC;
""", conn)

# ── Query 2: Top 20 Customers by Order Value ──────────────────────────────────
q2 = pd.read_sql("""
SELECT TOP 20
       C.custid,
       C.companyname,
       C.country,
       COUNT(DISTINCT O.orderid)                            AS OrderCount,
       SUM(OD.qty * OD.unitprice * (1 - OD.discount))      AS TotalRevenue
FROM   Sales.Customers   AS C
JOIN   Sales.Orders      AS O  ON C.custid    = O.custid
JOIN   Sales.OrderDetails AS OD ON O.orderid  = OD.orderid
GROUP  BY C.custid, C.companyname, C.country
ORDER  BY TotalRevenue DESC;
""", conn)

# ── Query 3: Monthly Revenue Trend ───────────────────────────────────────────
q3 = pd.read_sql("""
SELECT YEAR(O.orderdate)                                   AS OrderYear,
       MONTH(O.orderdate)                                  AS OrderMonth,
       SUM(OD.qty * OD.unitprice * (1 - OD.discount))     AS MonthlyRevenue,
       COUNT(DISTINCT O.orderid)                           AS OrderCount
FROM   Sales.Orders       AS O
JOIN   Sales.OrderDetails AS OD ON O.orderid = OD.orderid
GROUP  BY YEAR(O.orderdate), MONTH(O.orderdate)
ORDER  BY OrderYear, OrderMonth;
""", conn)

# ── Query 4: Employee Performance ────────────────────────────────────────────
q4 = pd.read_sql("""
SELECT E.empid,
       E.firstname + ' ' + E.lastname  AS EmployeeName,
       E.title,
       COUNT(O.orderid)                AS TotalOrders,
       SUM(O.freight)                  AS TotalFreight,
       AVG(O.freight)                  AS AvgFreight
FROM   HR.Employees   AS E
JOIN   Sales.Orders   AS O ON E.empid = O.empid
GROUP  BY E.empid, E.firstname, E.lastname, E.title
ORDER  BY TotalOrders DESC;
""", conn)

# ── Query 5: Product Sales Summary ───────────────────────────────────────────
q5 = pd.read_sql("""
SELECT P.productid,
       P.productname,
       P.categoryid,
       SUM(OD.qty)                                       AS UnitsSold,
       SUM(OD.qty * OD.unitprice * (1 - OD.discount))   AS Revenue,
       COUNT(DISTINCT OD.orderid)                        AS AppearancesInOrders
FROM   Production.Products  AS P
JOIN   Sales.OrderDetails   AS OD ON P.productid = OD.productid
GROUP  BY P.productid, P.productname, P.categoryid
ORDER  BY Revenue DESC;
""", conn)

conn.close()

# ── Write to Excel ────────────────────────────────────────────────────────────
output_path = 'NorthwindsPowerQuery_Report.xlsx'

sheets = {
    'Q1_OrdersByCountry':    q1,
    'Q2_TopCustomers':       q2,
    'Q3_MonthlyRevenue':     q3,
    'Q4_EmployeePerformance':q4,
    'Q5_ProductSales':       q5,
}

wb = openpyxl.Workbook()
wb.remove(wb.active)   # remove default blank sheet

HEADER_FILL  = PatternFill(fill_type='solid', fgColor='1F4E79')
HEADER_FONT  = Font(color='FFFFFF', bold=True)
HEADER_ALIGN = Alignment(horizontal='center')

for sheet_name, df in sheets.items():
    ws = wb.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), start=1):
        for c_idx, value in enumerate(row, start=1):
            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            if r_idx == 1:  # header row styling
                cell.fill      = HEADER_FILL
                cell.font      = HEADER_FONT
                cell.alignment = HEADER_ALIGN
    # Auto-fit column widths (approximate)
    for col in ws.columns:
        max_len = max((len(str(cell.value)) for cell in col if cell.value), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 40)

wb.save(output_path)
print(f'Excel report saved → {output_path}')
print('Open in Excel → select each sheet → Data → Get Data → From Table/Range → Power Query Editor')

---
## Summary of Propositions

| # | Proposition | DML Concept |
|---|-------------|-------------|
| 1 | Copy `Sales.Orders` into `dbo.Orders` | `SELECT INTO` |
| 2 | Insert single new customer row | `INSERT VALUES` |
| 3 | Insert customers who placed orders | `INSERT SELECT` + `EXISTS` |
| 4 | Delete pre-Aug-2014 orders with audit trail | `DELETE … OUTPUT` |
| 5 | Delete orders by Brazil customers via join | `DELETE … FROM JOIN` |
| 6 | Replace NULL regions with `<None>` | `UPDATE … OUTPUT` |
| 7 | Sync UK order shipping info from Customers | `UPDATE … FROM JOIN` |
| 8 | Upsert staging customers into main table | `MERGE … OUTPUT $action` |
| 9 | Load flat file into table | `BULK INSERT` |
| 10 | Empty child & parent tables efficiently | `TRUNCATE TABLE` |

---
*Assignment submitted per format: `Individual_GroupNumber_HW8_MemberName.ipynb`*